In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


In [2]:

# Load Microsoft stock data
df = pd.read_csv("../data/raw/msft.us.txt")

# Convert dates and sort chronologically
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Remove unused column
df = df.drop(columns=["OpenInt"])

# Handle zero-volume observations
df["Volume"] = df["Volume"].replace(0, np.nan).ffill()

# Engineer the same four features as previous models
df["Daily_Return"] = df["Close"].pct_change()

df["MA_5"] = df["Close"].rolling(5).mean() / df["Close"] - 1
df["MA_20"] = df["Close"].rolling(20).mean() / df["Close"] - 1

df["Volatility_5"] = df["Daily_Return"].rolling(5).std()

# Target: 1 if next trading day's closing price increases
df["Next_Close"] = df["Close"].shift(-1)
df["Target"] = (df["Next_Close"] > df["Close"]).astype(int)

# Remove incomplete observations
df = df.dropna(subset=[
    "Daily_Return",
    "MA_5",
    "MA_20",
    "Volatility_5",
    "Next_Close"
]).reset_index(drop=True)

# Define model features
features = [
    "Daily_Return",
    "MA_5",
    "MA_20",
    "Volatility_5"
]

X = df[features]
y = df["Target"]

print("Total observations:", len(df))
print("\nTarget distribution:")
print(y.value_counts(normalize=True))


Total observations: 7963

Target distribution:
Target
0    0.538239
1    0.461761
Name: proportion, dtype: float64


In [3]:

# Chronological 80/20 train/test split
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Fit scaler using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining feature shape:", X_train_scaled.shape)
print("Testing feature shape:", X_test_scaled.shape)


Training observations: 6370
Testing observations: 1593

Training feature shape: (6370, 4)
Testing feature shape: (1593, 4)


In [4]:

# Number of trading days in each sequence
LOOKBACK = 20

def create_sequences(X, y, lookback=20):
    X_sequences = []
    y_sequences = []

    for i in range(lookback - 1, len(X)):
        # Current day and previous 19 trading days
        X_sequences.append(X[i - lookback + 1:i + 1])

        # Target: next-day direction for current day
        y_sequences.append(y[i])

    return np.array(X_sequences), np.array(y_sequences)


# Training sequences
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train.to_numpy(),
    LOOKBACK
)

# Include the last 19 training days as test context
X_test_context = np.vstack([
    X_train_scaled[-(LOOKBACK - 1):],
    X_test_scaled
])

y_test_context = np.concatenate([
    y_train.to_numpy()[-(LOOKBACK - 1):],
    y_test.to_numpy()
])

# Testing sequences
X_test_seq, y_test_seq = create_sequences(
    X_test_context,
    y_test_context,
    LOOKBACK
)

print("Training sequences:", X_train_seq.shape)
print("Training targets:", y_train_seq.shape)

print("\nTesting sequences:", X_test_seq.shape)
print("Testing targets:", y_test_seq.shape)

# Verify target alignment
assert len(y_test_seq) == len(y_test)
assert np.array_equal(y_test_seq, y_test.to_numpy())

print("\nSequence alignment verified!")


Training sequences: (6351, 20, 4)
Training targets: (6351,)

Testing sequences: (1593, 20, 4)
Testing targets: (1593,)

Sequence alignment verified!


In [5]:

# Build LSTM model
model = Sequential([
    Input(shape=(LOOKBACK, len(features))),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

# Compile model for binary classification
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display model architecture
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,769 (18.63 KB)

 Trainable params: 4,769 (18.63 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:

# Chronological training/validation split
val_split_idx = int(len(X_train_seq) * 0.8)

X_fit = X_train_seq[:val_split_idx]
y_fit = y_train_seq[:val_split_idx]

X_val = X_train_seq[val_split_idx:]
y_val = y_train_seq[val_split_idx:]

print("Training sequences:", X_fit.shape)
print("Validation sequences:", X_val.shape)

# Stop training when validation loss stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Train the LSTM
history = model.fit(
    X_fit,
    y_fit,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)


Training sequences: (5080, 20, 4)
Validation sequences: (1271, 20, 4)
Epoch 1/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5465 - loss: 0.6900 - val_accuracy: 0.4980 - val_loss: 0.6936
Epoch 2/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5555 - loss: 0.6860 - val_accuracy: 0.4902 - val_loss: 0.6939
Epoch 3/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5567 - loss: 0.6870 - val_accuracy: 0.4870 - val_loss: 0.6937
Epoch 4/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5596 - loss: 0.6843 - val_accuracy: 0.4894 - val_loss: 0.6934
Epoch 5/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5616 - loss: 0.6846 - val_accuracy: 0.4902 - val_loss: 0.6936
Epoch 6/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5673 - loss: 0.6830 - val_accuracy: 0.4815 - val_loss: 0.6939
Epoch 7/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5620 - loss: 0.6829 - val_accuracy: 0.4902 - val_loss: 0.6941
Epoch 8/30
159/159 ━━━━━━━━━━━━━━━

In [7]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Predict probability of next-day price increase
y_pred_prob = model.predict(X_test_seq, verbose=0).ravel()

# Convert probabilities into binary predictions
y_pred = (y_pred_prob >= 0.5).astype(int)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test_seq, y_pred)
precision = precision_score(y_test_seq, y_pred, zero_division=0)
recall = recall_score(y_test_seq, y_pred, zero_division=0)
f1 = f1_score(y_test_seq, y_pred, zero_division=0)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_seq, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_seq, y_pred))

print("\nPredicted Class Distribution:")
print(pd.Series(y_pred).value_counts(normalize=True).sort_index())


Test Accuracy:  0.4915
Test Precision: 0.4891
Test Recall:    0.0557
Test F1 Score:  0.1000

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.94      0.65       785
           1       0.49      0.06      0.10       808

    accuracy                           0.49      1593
   macro avg       0.49      0.50      0.37      1593
weighted avg       0.49      0.49      0.37      1593


Confusion Matrix:
[[738  47]
 [763  45]]

Predicted Class Distribution:
0    0.942247
1    0.057753
Name: proportion, dtype: float64
